# Pipeline complet NdapuSpeech — exécution bout en bout

Ce notebook exécute **toutes les étapes** du pipeline ASR Wolof dans l'ordre :

| # | Étape | Durée estimée |
|---|-------|---------------|
| 1 | Téléchargement des datasets existants | ~10 min |
| 2 | Collecte YouTube (optionnel) | ~30 min |
| 3 | Construction du dataset unifié | ~5 min |
| 4 | Entraînement tokenizer BPE Wolof | ~2 min |
| 5 | Fine-tuning Whisper sur le Wolof | ~heures (GPU requis) |
| 6 | Entraînement KenLM n-gram | ~5 min |
| 7 | Évaluation WER/CER vs M-Kiriku | ~5 min |
| 8 | Inférence sur des fichiers audio | ~1 min |

> **Prérequis** : Python ≥ 3.11, GPU CUDA disponible (RTX recommandé),
> `ffmpeg` et `yt-dlp` installés dans le PATH.

## Prérequis — installation des packages

In [1]:
%pip install -e "C:/Users/ROG ZEPHYRUS G14/OneDrive - Université Cheikh Anta DIOP de DAKAR/Desktop/NdapuSpeech"[audio]

Obtaining file:///C:/Users/ROG%20ZEPHYRUS%20G14/OneDrive%20-%20Universit%C3%A9%20Cheikh%20Anta%20DIOP%20de%20DAKAR/Desktop/NdapuSpeech
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started
  Getting requirements to build editable: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'
  Building editable for ndapuspeech (pyproject.toml): started
  Building editable for ndapuspeech (pyproject.toml): finished with status 'done'
  Created wheel for ndapuspeech: filename=ndapuspeech-0.1.0-0.editable-py3-none-any.whl size=2881 sha256=8b9a6495a65a75d836ea2cdddcf9d1fff457892c92d190fbbb327e9c3fa3ba19
  Stored in directory: C:\Users\ROG ZEPHYRUS G14\AppDat

In [1]:
import logging, time, torch

logging.basicConfig(level=logging.INFO, format='%(message)s')
logging.getLogger().setLevel(logging.INFO)

from ndapuspeech import config, data, models, evaluation, inference
from ndapuspeech.training import TrainingConfig, train_model
from ndapuspeech.kenlm import train_kenlm

config.ensure_dirs()

print(f"CUDA : {torch.cuda.is_available()} — {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}")
print(f"Base model : {config.BASE_MODEL}")
print(f"Language : {config.LANGUAGE}")
print(f"Config chargée : {len(config.get_config())} sections")

CUDA : True — NVIDIA GeForce RTX 5060 Laptop GPU
Base model : openai/whisper-large-v3-turbo
Language : wo
Config chargée : 3 sections


---
## Étape 1 — Téléchargement des datasets

In [3]:
t0 = time.time()
results = data.download_all()
print(f"\n--- Téléchargements ({time.time()-t0:.0f}s) ---")
for source, path in results.items():
    status = 'OK' if path else 'SKIPPED'
    print(f"  {source:16s} {status}")

=== Kallaama (OpenSLR 151) — 55h agriculture ===
[SKIP] Kallaama already exists at C:\Users\ROG ZEPHYRUS G14\OneDrive - Université Cheikh Anta DIOP de DAKAR\Desktop\NdapuSpeech\data\raw\kallaama
=== ALFFA (OpenSLR 25) — 18h read speech ===
[SKIP] ALFFA already exists at C:\Users\ROG ZEPHYRUS G14\OneDrive - Université Cheikh Anta DIOP de DAKAR\Desktop\NdapuSpeech\data\raw\alffa
=== Google FLEURS Wolof — ~10h multi-domain ===
c:\Users\ROG ZEPHYRUS G14\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[SKIP] fleurs already exists at C:\Users\ROG ZEPHYRUS G14\OneDrive - Université Cheikh Anta DIOP de DAKAR\Desktop\NdapuSpeech\data\raw\fleurs
=== WolBanking77 — 4h banking ===
[SKIP] wolbanking77 already exists at C:\Users\ROG ZEPHYRUS G14\OneDrive - Université Cheikh Anta DIOP de D


--- Téléchargements (3s) ---
  kallaama         OK
  alffa            OK
  fleurs           OK
  wolbanking77     OK


---
## Étape 3 — Construction du dataset unifié train / val / test

In [4]:
t0 = time.time()
summary = data.build_unified_dataset(
    output_dir=config.PROCESSED_DATA_DIR / 'ndapuspeech_ds',
    train_ratio=config.TRAIN_SPLIT,
)
print(f"\nDataset ({time.time()-t0:.0f}s) :")
print(f"  Segments  : {summary['n_samples']}")
print(f"  Durée tot : {summary['total_hours']:.1f} h")
for src, hrs in sorted(summary['by_source'].items(), key=lambda x: -x[1]):
    print(f"  {src:20s} {hrs:6.1f} h")
print(f"  Splits    : {summary['splits']}")

[SKIP] Unified dataset already exists at C:\Users\ROG ZEPHYRUS G14\OneDrive - Université Cheikh Anta DIOP de DAKAR\Desktop\NdapuSpeech\data\processed\ndapuspeech_ds



Dataset (0s) :
  Segments  : 48903
  Durée tot : 80.9 h
  kallaama               48.3 h
  alffa                  21.4 h
  fleurs                 11.2 h
  Splits    : {'train': 41567, 'val': 4888, 'test': 2448}


---
## Étape 4 — Entraînement du tokenizer BPE Wolof

In [5]:
t0 = time.time()
tok_result = models.train_tokenizer_pipeline(
    vocab_size=config.DATA['tokenizer']['vocab_size'],
    extend_whisper=True,
)
print(f"\nTokenizer ({time.time()-t0:.0f}s) :")
print(f"  Pièces BPE   : {tok_result['n_pieces']}")
print(f"  Corpus chars  : {tok_result['corpus_chars']}")
print(f"  Étendu vers   : {tok_result['extended_tokenizer']}")

HTTP Request: HEAD https://huggingface.co/openai/whisper-large-v3-turbo/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/openai/whisper-large-v3-turbo/41f01f3fe87f28c78e2fbf8b568835947dd65ed9/config.json "HTTP/1.1 200 OK"
HTTP Request: HEAD https://huggingface.co/openai/whisper-large-v3-turbo/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/openai/whisper-large-v3-turbo/41f01f3fe87f28c78e2fbf8b568835947dd65ed9/tokenizer_config.json "HTTP/1.1 200 OK"
HTTP Request: GET https://huggingface.co/api/models/openai/whisper-large-v3-turbo/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
HTTP Request: GET https://huggingface.co/api/models/openai/whisper-large-v3-turbo/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
Tokenizer saved to C:\Users\ROG ZEPHYRUS G14\OneDrive - Université Cheik


Tokenizer (538s) :
  Pièces BPE   : 9997
  Corpus chars  : 5201475
  Étendu vers   : C:\Users\ROG ZEPHYRUS G14\OneDrive - Université Cheikh Anta DIOP de DAKAR\Desktop\NdapuSpeech\models\whisper-tokenizer-extended


---
## Étape 5 — Fine-tuning Whisper sur le Wolof

Pour un vrai entraînement, supprimez les lignes de `num_epochs`/`max_steps`
et laissez les valeurs par défaut dans `configs/training.yaml`.

In [2]:
cfg = TrainingConfig.from_yaml({
    'num_epochs': 20,
    'eval_steps': 200,
    'save_steps': 400,
})
print(cfg)

TrainingConfig(data_dir=WindowsPath('C:/Users/ROG ZEPHYRUS G14/OneDrive - Université Cheikh Anta DIOP de DAKAR/Desktop/NdapuSpeech/data/processed/ndapuspeech_ds'), output_dir=WindowsPath('C:/Users/ROG ZEPHYRUS G14/OneDrive - Université Cheikh Anta DIOP de DAKAR/Desktop/NdapuSpeech/models/ndapuspeech-v1'), base_model='openai/whisper-large-v3-turbo', tokenizer_dir='C:\\Users\\ROG ZEPHYRUS G14\\OneDrive - Université Cheikh Anta DIOP de DAKAR\\Desktop\\NdapuSpeech\\models\\whisper-tokenizer-extended', num_epochs=20, batch_size=4, grad_accum=4, lr_decoder=2e-05, lr_encoder=1e-05, warmup_steps=100, patience=5, noise_dir=None, use_lora=True, resume_from=None, max_steps=None, eval_steps=200, save_steps=400, seed=42, push_to_hub=None, fp16=True, bf16=False, hub_dataset='ndapuspeech/ndapuspeech-dataset')


In [ ]:
t0 = time.time()
out_dir = train_model(cfg)
print(f"\nModèle sauvegardé : {out_dir} ({(time.time()-t0)/60:.0f} min)")

c:\Users\ROG ZEPHYRUS G14\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Device: cuda
GPU: NVIDIA GeForce RTX 5060 Laptop GPU
HTTP Request: GET https://huggingface.co/api/models/openai/whisper-large-v3-turbo/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
HTTP Request: HEAD https://huggingface.co/openai/whisper-large-v3-turbo/resolve/main/processor_config.json "HTTP/1.1 404 Not Found"
HTTP Request: HEAD https://huggingface.co/openai/whisper-large-v3-turbo/resolve/main/chat_template.json "HTTP/1.1 404 Not Found"
HTTP Request: HEAD https://huggingface.co/openai/whisper-large-v3-turbo/resolve/main/chat_template.jinja "HTTP/1.1 404 Not Found"
HTTP Request: HEAD https://huggingface.co/openai/whisper-large-v3-turbo/resolve/main/audio_toke

trainable params: 13,107,200 || all params: 833,190,400 || trainable%: 1.5731


HTTP Request: HEAD https://s3.amazonaws.com/datasets.huggingface.co/datasets/datasets/json/json.py "HTTP/1.1 200 OK"
Generating train split: 41567 examples [00:00, 590174.52 examples/s]
Generating val split: 4888 examples [00:00, 361704.24 examples/s]
Generating test split: 2448 examples [00:00, 222793.39 examples/s]
Preparing val dataset (num_proc=1): 100%|██████████| 4888/4888 [02:21<00:00, 34.47 examples/s]
Train samples: 41567, Val samples: 4888


Step,Training Loss,Validation Loss


[transformers] Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
[transformers] Transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English. This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`. See https://github.com/huggingface/transformers/pull/28687 for more details.
[transformers] Passing `generation_config` together with generation-related arguments=({'begin_suppress_tokens', 'suppress_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] The attention mask is not set with a batched input, and cannot be inferred from input because pad token is same as eos token. As a consequence, you may obser

---
## Étape 6 — Entraînement KenLM n-gram

In [ ]:
arpa_path, bin_path = train_kenlm()
print(f"ARPA : {arpa_path}")
print(f"BIN  : {bin_path}")

---
## Étape 7 — Évaluation WER / CER vs M-Kiriku

In [ ]:
from ndapuspeech.data import load_jsonl

test_samples = load_jsonl(config.PROCESSED_DATA_DIR / 'ndapuspeech_ds' / 'test.jsonl')
print(f'{len(test_samples)} échantillons de test')

summaries = evaluation.evaluate_and_compare(
    models=[
        str(out_dir),
        'AIHubSN/M-Kiriku-ASR',
    ],
    samples=test_samples,
    batch_size=8,
)

In [ ]:
import pandas as pd

rows = [
    {'modèle': m, 'WER': s['overall']['wer'], 'CER': s['overall']['cer']}
    for m, s in summaries.items()
]
df = pd.DataFrame(rows)
display(df)

baseline_wer = summaries.get('AIHubSN/M-Kiriku-ASR', {}).get('overall', {}).get('wer', None)
ours_wer = summaries.get(str(out_dir), {}).get('overall', {}).get('wer', None)
if baseline_wer and ours_wer:
    verdict = 'BEATS baseline' if ours_wer < baseline_wer else 'sous la baseline'
    print(f"\nVerdict : {verdict}")
    print(f"  M-Kiriku : {baseline_wer:.2f}%")
    print(f"  NdapuSpeech : {ours_wer:.2f}%")

---
## Étape 8 — Inférence sur des fichiers audio

In [ ]:
import glob

audio_files = sorted(glob.glob('data/youtube/segments/*.wav'))[:5]
if not audio_files:
    print('Aucun fichier audio trouvé dans data/youtube/segments/')
else:
    results = inference.transcribe_files(
        model_path=str(out_dir),
        audio_paths=audio_files,
        lm_path=str(bin_path) if bin_path else None,
    )
    for path, r in results.items():
        print(f'\n--- {path} ---')
        print(r['text'])

---
## Résumé

Le pipeline complet est terminé. Pour un entraînement sérieux :

1. Mettez `num_epochs=20` (ou plus) dans `configs/training.yaml`
2. Activez la collecte YouTube (décommentez l'étape 2)
3. Relancez ce notebook sur un GPU puissant (≥ 16 Go VRAM)
4. Poussez le modèle sur HuggingFace Hub

```python
# Push vers Hub (décommenter)
# from huggingface_hub import login; login()
# model.push_to_hub('ndapuspeech/ndapuspeech-asr')
# processor.push_to_hub('ndapuspeech/ndapuspeech-asr')
```